# Sprint 4 — PLN: Pipeline RAG e Assistente Conversacional de Troubleshooting

**Challenge FIAP — Forzy | Processamento de Linguagem Natural**
Solução de *digital-twin* para monitoramento e manutenção preditiva de motores elétricos industriais.

## Entregáveis cobertos por este notebook

| Etapa | Entregável | Célula |
|---|---|---|
| 1 | Chunking inteligente da documentação técnica, preservando coerência semântica | 4 |
| 2 | Geração de embeddings e indexação em base vetorial (ChromaDB) | 5 |
| 3 | Retriever com re-ranking por relevância contextual (CrossEncoder) | 6 |
| 4 | Assistente conversacional: persona técnica, memória de curto prazo e injeção do estado operacional | 7 |
| 5 | Demonstração nos cenários de falha | 8 |
| 6 | Avaliação — faithfulness, answer relevancy e context precision (LLM-as-a-judge, 20 perguntas) | 9 e 10 |
| 7 | Limites do sistema e estratégias de mitigação | 11 |

**Documento indexado:** `docs/WEG-w22-motor-eletrico-trifasico-brochure.pdf`

**Embeddings:** `all-MiniLM-L6-v2` · **Re-ranking:** `cross-encoder/ms-marco-MiniLM-L-6-v2` · **LLM:** Groq (`qwen/qwen3.8-27b`, com fallback para `llama3-8b-8192` e OpenRouter).

> Execute as células na ordem, de cima para baixo. A célula 5 baixa os modelos de embedding na primeira execução.

## 1. Dependências

Instale as bibliotecas necessárias (basta executar uma vez por ambiente).

In [1]:
%pip install -q chromadb sentence-transformers PyMuPDF python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuração do ambiente

Carrega as chaves de API a partir do arquivo `.env` na raiz do projeto (`NLP/.env`), que deve conter:

```
GROQ-API-KEY=sua_chave_aqui
OPEN-ROUTER-API-KEY=sua_chave_aqui
```

A chave do OpenRouter é opcional — ela serve apenas como **fallback de provedor** quando a Groq
retorna *rate limit* (HTTP 429).

In [2]:
import json
import os
import re
import time
from pathlib import Path

import chromadb
import fitz  # PyMuPDF
import requests
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder, SentenceTransformer


# Resolve a raiz do projeto (pasta NLP) a partir do diretório de trabalho atual,
# funcionando tanto ao executar de notebooks/ quanto da raiz do repositório.
def resolver_base_dir():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "docs").is_dir() and (candidato / "data").is_dir():
            return candidato
    return Path.cwd()


BASE_DIR = resolver_base_dir()
PDF_PATH = BASE_DIR / "docs" / "WEG-w22-motor-eletrico-trifasico-brochure.pdf"

load_dotenv(BASE_DIR / ".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("GROQ-API-KEY")
OPENROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY") or os.getenv("OPEN-ROUTER-API-KEY")

print(f"Raiz do projeto: {BASE_DIR}")
print(f"Documento técnico: {PDF_PATH} (existe: {PDF_PATH.exists()})")
print(f"Chave da Groq carregada: {bool(GROQ_API_KEY)}")
print(f"Chave do OpenRouter (fallback) carregada: {bool(OPENROUTER_API_KEY)}")

c:\Users\Igor\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raiz do projeto: c:\Users\Igor\Desktop\Sarak\Fiap\CP - Sprint - GS\2º Ano\Sprints\Sprint 3\NLP
Documento técnico: c:\Users\Igor\Desktop\Sarak\Fiap\CP - Sprint - GS\2º Ano\Sprints\Sprint 3\NLP\docs\WEG-w22-motor-eletrico-trifasico-brochure.pdf (existe: True)
Chave da Groq carregada: True
Chave do OpenRouter (fallback) carregada: True


## 3. Chunking inteligente da documentação técnica

Em vez de cortar o documento em janelas de tamanho fixo — o que quebra frases e separa um
cabeçalho técnico do seu conteúdo —, o texto é segmentado **por parágrafo** e os parágrafos são
agrupados até o limite de 1000 caracteres.

Isso preserva a coerência semântica de cada chunk: um procedimento de manutenção ou uma
especificação técnica chega inteiro ao índice vetorial.

In [3]:
def intelligent_chunk_text(text, max_chunk_size=1000):
    """Quebra o documento em parágrafos para não cortar frases ao meio."""
    paragraphs = re.split(r'\n\s*\n', text)
    chunks = []
    current_chunk = ""

    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if len(current_chunk) + len(p) <= max_chunk_size:
            current_chunk += p + "\n\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = p + "\n\n"

    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

## 4. Retriever com re-ranking por relevância contextual

A recuperação acontece em dois passos:

1. **Busca vetorial ampla** — o ChromaDB retorna os `top_k = 5` chunks mais próximos do embedding
   da pergunta (recall alto, precisão moderada);
2. **Re-ranking com CrossEncoder** — cada par *(pergunta, chunk)* é reavaliado por um modelo que
   lê os dois textos juntos, e apenas os `final_k = 2` melhores chegam ao LLM.

O segundo passo é o que eleva a **precisão dos chunks entregues** ao assistente: o bi-encoder da
busca vetorial compara vetores independentes, enquanto o cross-encoder julga a relevância real do
trecho para aquela pergunta específica.

In [4]:
class RAGRetriever:
    def __init__(self, collection):
        self.collection = collection
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        print("Carregando modelo de Re-ranking (CrossEncoder)...")
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def invoke(self, query, top_k=5, final_k=2):
        # Passo 1: Recuperação inicial ampla
        query_embedding = self.model.encode(query).tolist()
        results = self.collection.query(query_embeddings=[query_embedding], n_results=top_k)

        docs = results['documents'][0] if results['documents'] else []
        if not docs:
            return []

        # Passo 2: Re-ranking usando CrossEncoder
        pairs = [[query, doc] for doc in docs]
        scores = self.cross_encoder.predict(pairs)

        scored_docs = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)

        # Retorna apenas os 'final_k' documentos mais relevantes
        return [doc for score, doc in scored_docs[:final_k]]

## 5. Ingestão do documento, embeddings e indexação vetorial

Extrai o texto do PDF página a página (PyMuPDF), aplica o chunking semântico, gera os embeddings
com `all-MiniLM-L6-v2` e indexa tudo em uma coleção do **ChromaDB**.

> A primeira execução baixa os modelos de embedding e de re-ranking (algumas centenas de MB).

In [5]:
def get_retriever():
    print(f"Carregando documento PDF: {PDF_PATH}")
    doc = fitz.open(PDF_PATH)
    full_text = ""
    for page in doc:
        extracted = page.get_text()
        if extracted:
            full_text += extracted + "\n"

    print("Realizando chunking inteligente...")
    splits = intelligent_chunk_text(full_text)
    print(f"Total de chunks (semânticos): {len(splits)}")

    print("Gerando embeddings e vectorstore...")
    chroma_client = chromadb.Client()
    try:
        chroma_client.delete_collection("weg_manual")
    except Exception:
        pass

    collection = chroma_client.create_collection(name="weg_manual")

    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(splits).tolist()

    ids = [str(i) for i in range(len(splits))]
    collection.add(
        embeddings=embeddings,
        documents=splits,
        ids=ids
    )
    return RAGRetriever(collection)


if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF não encontrado no caminho: {PDF_PATH}")

retriever = get_retriever()
print("Retriever pronto.")

Carregando documento PDF: c:\Users\Igor\Desktop\Sarak\Fiap\CP - Sprint - GS\2º Ano\Sprints\Sprint 3\NLP\docs\WEG-w22-motor-eletrico-trifasico-brochure.pdf
Realizando chunking inteligente...
Total de chunks (semânticos): 77
Gerando embeddings e vectorstore...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5982.40it/s]


Carregando modelo de Re-ranking (CrossEncoder)...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4020.14it/s]


Retriever pronto.


## 6. Assistente conversacional de troubleshooting

Integra o retriever ao LLM. O prompt de sistema reúne quatro blocos:

- **Persona** — engenheiro especialista em manutenção de motores elétricos, instruído a citar
  trechos do manual e a **indicar o nível de confiança** da resposta;
- **Contexto técnico recuperado** — os chunks re-ranqueados, que fundamentam a resposta;
- **Estado atual da máquina** — o contexto operacional real vindo da Sprint 3 (resumo de alerta e
  telemetria), permitindo respostas sensíveis à situação do ativo;
- **Histórico recente** — memória de curto prazo com as duas últimas interações, para manter o fio
  do diálogo de troubleshooting.

O envio ao LLM tem **fallback em cascata**: `qwen/qwen3.8-27b` → `llama3-8b-8192` na Groq e, em caso
de *rate limit* nos dois, OpenRouter como provedor alternativo.

In [6]:
class ConversationalAssistant:
    def __init__(self, retriever):
        self.retriever = retriever
        self.history = []

    def chat(self, query, state_context=""):
        retrieved_docs = self.retriever.invoke(query)
        context = "\n\n".join(retrieved_docs)

        # Injeção de histórico no prompt (Memória de curto prazo - últimas 2 interações)
        hist_text = "\n".join([f"Usuário: {u}\nAssistente: {a}" for u, a in self.history[-2:]])
        if hist_text:
            hist_text = "Histórico Recente:\n" + hist_text + "\n"

        prompt = f"""Você é um Engenheiro Especialista em Manutenção de Motores Elétricos.
Use o contexto técnico fornecido abaixo para responder a pergunta. Cite trechos do manual, se possível.

Contexto Técnico do Manual Recuperado:
{context}

Estado Atual (Telemetria) da Máquina:
{state_context}

{hist_text}
Pergunta Atual do Usuário: {query}
Responda com confiança, assertividade e indique seu nível de confiança na resposta baseada no documento."""

        headers = {
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        }
        # Tenta modelos na Groq primeiro
        models_groq = ["qwen/qwen3.8-27b", "llama3-8b-8192"]
        for model in models_groq:
            payload = {
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.2
            }
            try:
                resp = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
                resp.raise_for_status()
                answer = resp.json()["choices"][0]["message"]["content"]
                self.history.append((query, answer))
                return answer, context
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 429:
                    print(f"[Aviso] Rate limit Groq ({model})...")
                    continue
            except Exception:
                continue

        # Fallback de Provedor: OpenRouter
        if OPENROUTER_API_KEY:
            print("[Aviso] Alternando para OpenRouter (Fallback de Provedor)...")
            headers_or = {
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json"
            }
            payload_or = {
                "model": "openrouter/free",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.2
            }
            try:
                resp = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers_or, json=payload_or)
                resp.raise_for_status()
                answer = resp.json()["choices"][0]["message"]["content"]
                self.history.append((query, answer))
                return answer, context
            except Exception as e:
                return f"Erro em todos os provedores (Groq e OpenRouter). Último erro: {e}", context

        return "Erro: Falha na Groq e OpenRouter não configurado.", context


assistant = ConversationalAssistant(retriever)
print("Assistente conversacional pronto.")

Assistente conversacional pronto.


## 7. Demonstração — cenário de anomalia térmica com injeção do estado operacional

O estado operacional real do ativo (produzido pela Sprint 3) entra no prompt como contexto
adicional: o assistente responde sabendo que **há um alerta crítico de 47 °C pendente naquele
motor**, e não apenas o que o manual diz em abstrato.

In [7]:
print("\n>>> Cenário 1: Temperatura Crítica (Injeção de Estado)")

estado_1 = "Temperatura crítica registrada (47°C) - Falha pendente"
p1 = "A temperatura do motor disparou para 47°C, o alarme tocou. O que eu faço?"

print(f"Pergunta: {p1}")
r1, _ = assistant.chat(p1, estado_1)
print(f"Resposta:\n{r1}")


>>> Cenário 1: Temperatura Crítica (Injeção de Estado)
Pergunta: A temperatura do motor disparou para 47°C, o alarme tocou. O que eu faço?
Resposta:
Olá. Como Engenheiro Especialista em Manutenção de Motores Elétricos, analisei a situação reportada à luz do manual técnico do motor W22.

**Diagnóstico Inicial e Correção Técnica:**
Há uma **inconsistência crítica** entre o valor reportado (47°C) e a ação descrita (alarme/falha pendente).
*   **47°C é uma temperatura ambiente normal ou ligeiramente elevada**, mas **não é uma temperatura crítica** para um motor elétrico. Motores W22 possuem isolação de Classe F (resistência a 155°C) ou H (180°C).
*   O disparo de alarme/falha em 47°C indica, com alta probabilidade, uma **falha no sensor de temperatura** (termistor PTC, Pt-1000 ou bimetálico) ou uma **falha de comunicação/leitura no inversor de frequência ou no relé de proteção (RPW-PTC)**, e não um superaquecimento real do enrolamento.

---

### **Plano de Ação Imediato (Passo a Passo)**


### 7.1. Memória de curto prazo — continuidade do diálogo

A pergunta seguinte não repete o assunto ("*desse problema que você acabou de citar*").
A resposta correta só é possível porque o histórico da interação anterior foi reinjetado no prompt.

In [8]:
print("\n>>> Cenário 2: Teste de Memória de Curto Prazo (Diálogo)")

p2 = "E quais as possíveis causas raiz desse problema que você acabou de citar?"

print(f"Pergunta: {p2}")
r2, _ = assistant.chat(p2, estado_1)
print(f"Resposta:\n{r2}")


>>> Cenário 2: Teste de Memória de Curto Prazo (Diálogo)
Pergunta: E quais as possíveis causas raiz desse problema que você acabou de citar?
[Aviso] Rate limit Groq (qwen/qwen3.8-27b)...
[Aviso] Alternando para OpenRouter (Fallback de Provedor)...
Resposta:
### Possíveis Causas Raiz do Falso Alarme de Temperatura em 47°C  
Baseando-se estritamente no **contexto técnico do manual W22 fornecido** e na análise anterior, as causas raiz para o disparo de alarme/falha em 47°C (temperatura incompatível com superaquecimento real) são as seguintes. Cada causa está vinculada a trechos específicos do manual, garantindo assertividade técnica:

---

#### **1. Falha no Sensor de Temperatura**  
O manual descreve explicitamente os tipos de protetores térmicos utilizados e seu comportamento esperado, permitindo identificar falhas características:  
- **Bimetálico (tipo NF)**:  
  > *"Os protetores térmicos do tipo bimetálico são protetores térmicos com contatos de prata, tipo NF (normalmente fechados

## 8. Avaliação — LLM-as-a-judge

Um segundo LLM, atuando como juiz, pontua cada resposta de 1 a 5 em duas dimensões:

- **faithfulness** — a resposta é sustentada pelo contexto recuperado, sem alucinar ou inventar dados?
- **answer relevancy** — a resposta atende à pergunta, comparada ao gabarito de referência?

A resposta do juiz é forçada ao formato JSON (`response_format`), com extração por regex como
salvaguarda quando o provedor de fallback não respeita o modo JSON nativamente.

In [9]:
def evaluate_rag(query, answer, context, ground_truth):
    prompt = f"""Avalie a resposta RAG gerada baseada nos seguintes critérios, retornando APENAS um JSON válido e nada mais.
Critérios:
1. faithfulness: A resposta (answer) é fiel ao contexto (context) fornecido, sem alucinar ou inventar dados? (Dar nota de 1 a 5)
2. answer_relevancy: A resposta atende e resolve a pergunta (query) baseada na referência (ground_truth)? (Dar nota de 1 a 5)

Pergunta: {query}
Contexto Recuperado: {context}
Resposta Gerada: {answer}
Gabarito de Referência (Ground Truth): {ground_truth}

Retorne ESTRITAMENTE o formato JSON:
{{"faithfulness": score, "answer_relevancy": score, "justificativa": "sua justificativa breve"}}"""

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    models_groq = ["qwen/qwen3.8-27b", "llama3-8b-8192"]

    for model in models_groq:
        payload = {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "response_format": {"type": "json_object"}
        }

        try:
            resp = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"]
            return json.loads(content)
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429:
                continue
        except Exception:
            continue

    if OPENROUTER_API_KEY:
        headers_or = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        }
        payload_or = {
            "model": "openrouter/free",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1
        }
        try:
            resp = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers_or, json=payload_or)
            resp.raise_for_status()

            content = resp.json()["choices"][0]["message"]["content"]
            # OpenRouter nem sempre respeita o json_object nativamente; extrai o JSON do texto.
            try:
                return json.loads(content)
            except Exception:
                match = re.search(r'\{.*\}', content, re.DOTALL)
                if match:
                    return json.loads(match.group(0))
                return {"faithfulness": 0, "answer_relevancy": 0, "justificativa": "JSON inválido do OpenRouter"}
        except Exception as e:
            return {"faithfulness": 0, "answer_relevancy": 0, "justificativa": f"Erro Fallback OpenRouter: {e}"}

    return {"faithfulness": 0, "answer_relevancy": 0, "justificativa": "Erro: Limite de taxa na Groq. OpenRouter indisponível."}

## 9. Conjunto de avaliação — 20 perguntas de troubleshooting

Perguntas com respostas de referência (*ground truth*) extraídas do manual técnico do ativo.
O histórico é limpo antes de cada pergunta, para que cada item seja avaliado isoladamente, e há
uma pausa entre as chamadas para respeitar o limite de taxa da API.

In [10]:
def run_evaluation(assistant):
    perguntas_teste = [
        {"q": "O que indica a proteção térmica PTC?", "gt": "Indica sobreaquecimento, variando resistência e desligando o circuito principal."},
        {"q": "Qual a classe de isolamento padrão do W22?", "gt": "O motor W22 padrão possui classe de isolamento F."},
        {"q": "Qual o limite de temperatura ambiente para operação?", "gt": "-30°C a +40°C."},
        {"q": "Como é a proteção contra correntes de mancal para carcaças 315S/M?", "gt": "Uso de rolamento isolado ou tampa com cubo isolado e escova de aterramento."},
        {"q": "Para que serve a graxa nos rolamentos?", "gt": "Para lubrificação e redução de atrito e temperatura."},
        {"q": "Os motores com rendimento IR3 podem operar com inversor de frequência?", "gt": "Sim, são otimizados para operar com inversores."},
        {"q": "Qual o grau de proteção dos motores padrão?", "gt": "Grau de proteção IP55."},
        {"q": "Qual fator afeta a vida útil dos rolamentos?", "gt": "Operação em velocidades variadas ou desalinhamento mecânico."},
        {"q": "Quais cores são usadas no esquema de pintura?", "gt": "Azul RAL 5009 padrão."},
        {"q": "O que deve ser feito se a tensão estiver fora do padrão?", "gt": "Usar relés de proteção para evitar sobrecarga."},
        {"q": "Como funciona o dreno de condensação?", "gt": "Os drenos permitem a saída de água condensada e devem ser abertos periodicamente."},
        {"q": "O motor possui resistência de aquecimento?", "gt": "Opcional, usada para prevenir condensação durante paradas prolongadas."},
        {"q": "Quando usar rolamentos de rolo?", "gt": "Recomendados para aplicações com altas cargas radiais."},
        {"q": "Qual material da carcaça do motor W22?", "gt": "Ferro fundido FC-200."},
        {"q": "Qual a vantagem das aletas na tampa dianteira?", "gt": "Melhora a ventilação e reduz a temperatura do rolamento."},
        {"q": "Como é o balanceamento do eixo?", "gt": "O motor é balanceado dinamicamente com meia chaveta padrão."},
        {"q": "Para qual frequência os motores são projetados?", "gt": "Opcionalmente 50Hz ou 60Hz."},
        {"q": "O que acontece se o motor vibrar excessivamente?", "gt": "Pode haver desbalanceamento mecânico exigindo parada imediata."},
        {"q": "O que é derating de potência?", "gt": "Redução da potência útil se temperatura ambiente > 40°C ou altitude > 1000m."},
        {"q": "Qual a função do anel V-ring?", "gt": "Promover vedação contra poeira e água no eixo."}
    ]

    print("\n" + "=" * 60)
    print(" INICIANDO AVALIAÇÃO RAGAS (LLM-as-a-Judge) - 20 PERGUNTAS")
    print("=" * 60)

    total_faithfulness = 0
    total_relevancy = 0
    num_questions = len(perguntas_teste)

    for i, item in enumerate(perguntas_teste):
        print(f"\n[Avaliando {i + 1}/{num_questions}] P: {item['q']}")
        # Limpa o histórico antes de avaliar cada pergunta solta do dataset
        assistant.history = []
        ans, ctx = assistant.chat(item['q'], state_context="Condições normais.")

        eval_metrics = evaluate_rag(item['q'], ans, ctx, item['gt'])

        f_score = eval_metrics.get('faithfulness', 0)
        r_score = eval_metrics.get('answer_relevancy', 0)

        total_faithfulness += f_score
        total_relevancy += r_score

        print(f"   --> Faithfulness: {f_score}/5 | Answer Relevancy: {r_score}/5")
        print(f"   --> Justificativa: {eval_metrics.get('justificativa', '')}")
        time.sleep(3)  # Tempo de espera para garantir respiro à API

    print("\n" + "=" * 60)
    print(" RESULTADOS FINAIS DA AVALIAÇÃO RAG")
    print(f" Média Faithfulness:     {total_faithfulness / num_questions:.2f} / 5.0")
    print(f" Média Answer Relevancy: {total_relevancy / num_questions:.2f} / 5.0")
    print("=" * 60)

### 9.1. Execução da suíte de avaliação

> Esta célula faz cerca de 40 chamadas de API com pausa de 3 segundos entre as perguntas —
> a execução completa leva alguns minutos.

In [11]:
run_evaluation(assistant)


 INICIANDO AVALIAÇÃO RAGAS (LLM-as-a-Judge) - 20 PERGUNTAS

[Avaliando 1/20] P: O que indica a proteção térmica PTC?
   --> Faithfulness: 5/5 | Answer Relevancy: 4/5
   --> Justificativa: A resposta é totalmente fiel ao contexto, citando corretamente as causas de sobreaquecimento e as funções de alarme/desligamento. Em relação à relevância, ela atende bem à pergunta, mas o gabarito específico menciona o mecanismo de 'variação de resistência' e 'desligamento do circuito principal', detalhes que a resposta gerada aborda de forma mais genérica (focando nas causas e no uso para alarme/desligamento), embora não contradiga o contexto.

[Avaliando 2/20] P: Qual a classe de isolamento padrão do W22?
   --> Faithfulness: 4/5 | Answer Relevancy: 2/5
   --> Justificativa: A resposta é majoritariamente fiel ao contexto, pois corretamente identifica que a informação sobre classe de isolamento não está presente nos trechos recuperados do manual W22. No entanto, inclui uma menção externa às classes 

## 10. Limites identificados e estratégias de mitigação

| Limite | Como se manifesta | Mitigação adotada |
|---|---|---|
| **Perguntas fora do escopo** | Questões sobre ativos ou temas não cobertos pelo PDF indexado. | O re-ranking descarta chunks irracionais e a persona exige que o modelo declare o nível de confiança, sinalizando quando a resposta não se apoia no documento. |
| **Alucinação de valores numéricos** | O modelo pode preencher lacunas com números plausíveis mas ausentes do manual. | Temperatura baixa (0.2), instrução explícita para citar trechos do manual e avaliação de *faithfulness* que penaliza afirmações sem lastro no contexto. |
| **Perda de contexto no diálogo** | Respostas descoladas do assunto após várias trocas. | Memória de curto prazo com as duas últimas interações reinjetadas no prompt. |
| **Chunking de PDF com layout complexo** | Tabelas e colunas do brochure podem ser extraídas fora de ordem. | Segmentação por parágrafo (e não por janela fixa), preservando blocos semânticos completos. |
| **Rate limit e indisponibilidade do provedor** | HTTP 429 interrompe a avaliação em lote. | Cascata de modelos na Groq, fallback para o OpenRouter e pausa entre as chamadas. |
| **Juiz LLM como métrica** | O avaliador é o mesmo tipo de modelo avaliado, o que pode enviesar as notas. | Gabarito manual (*ground truth*) fornecido ao juiz e justificativa exigida em cada nota, tornando a pontuação auditável.

## Conclusão

Este notebook entregou o pipeline RAG completo sobre a documentação técnica do ativo:

- **chunking semântico** por parágrafo, preservando a coerência dos blocos técnicos;
- **embeddings e indexação** em base vetorial ChromaDB;
- **retriever com re-ranking** por CrossEncoder, elevando a precisão dos chunks entregues ao LLM;
- **assistente conversacional** com persona de engenheiro especialista, citação de fonte, nível de
  confiança, memória de curto prazo e injeção do estado operacional real gerado na Sprint 3;
- **avaliação** de faithfulness e answer relevancy sobre 20 perguntas com respostas de referência;
- **documentação dos limites** do sistema e das estratégias de mitigação adotadas.